# SketchCritic Model Evaluation

This notebook evaluates the saved MLP and Random Forest artifacts on the synthetic CSV.

In [1]:
from pathlib import Path
import pickle

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "sketchcritic_synthetic.csv"
MLP_PATH = PROJECT_ROOT / "models" / "sketchcritic_mlp.pkl"
RF_PATH = PROJECT_ROOT / "models" / "sketchcritic_rf.pkl"

DATA_PATH, MLP_PATH, RF_PATH

(WindowsPath('c:/Users/rache/OneDrive/Desktop/Duke/cs 372/Final Project/SketchCritic/data/sketchcritic_synthetic.csv'),
 WindowsPath('c:/Users/rache/OneDrive/Desktop/Duke/cs 372/Final Project/SketchCritic/models/sketchcritic_mlp.pkl'),
 WindowsPath('c:/Users/rache/OneDrive/Desktop/Duke/cs 372/Final Project/SketchCritic/models/sketchcritic_rf.pkl'))

In [3]:
data = pd.read_csv(DATA_PATH)
label_columns = [column for column in data.columns if column.startswith("label_")]
feature_columns = [column for column in data.columns if column not in label_columns and column != "active_labels"]

X = data[feature_columns]
y = data[label_columns]

print(f"Rows: {len(data)}")
print(f"Features: {len(feature_columns)}")
print(f"Labels: {len(label_columns)}")

Rows: 194
Features: 22
Labels: 20


In [4]:
def load_artifact(path: Path) -> dict:
    with path.open("rb") as handle:
        return pickle.load(handle)


def evaluate_artifact(path: Path) -> dict:
    artifact = load_artifact(path)
    model = artifact["model"]
    feature_names = artifact["feature_names"]
    label_names = artifact["label_names"]

    X_eval = data[feature_names]
    y_eval = data[label_names]
    y_pred = model.predict(X_eval)

    return {
        "model_path": str(path),
        "exact_match_accuracy": accuracy_score(y_eval, y_pred),
        "micro_f1": f1_score(y_eval, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_eval, y_pred, average="macro", zero_division=0),
        "classification_report": classification_report(
            y_eval,
            y_pred,
            target_names=label_names,
            zero_division=0,
        ),
    }

In [5]:
mlp_results = evaluate_artifact(MLP_PATH)
rf_results = evaluate_artifact(RF_PATH)

summary = pd.DataFrame([
    {
        "model": "MLP",
        "exact_match_accuracy": mlp_results["exact_match_accuracy"],
        "micro_f1": mlp_results["micro_f1"],
        "macro_f1": mlp_results["macro_f1"],
    },
    {
        "model": "Random Forest",
        "exact_match_accuracy": rf_results["exact_match_accuracy"],
        "micro_f1": rf_results["micro_f1"],
        "macro_f1": rf_results["macro_f1"],
    },
])

summary

,model,exact_match_accuracy,micro_f1,macro_f1
0,MLP,0.025773,0.128312,0.073984
1,Random Forest,0.943299,0.979206,0.964017


In [6]:
print("MLP classification report")
print(mlp_results["classification_report"])

print("Random Forest classification report")
print(rf_results["classification_report"])

MLP classification report
                                                 precision    recall  f1-score   support

                                 label_balanced       0.00      0.00      0.00         4
          label_middle_lower_vertical_imbalance       0.10      0.62      0.18         8
                          label_nose_misaligned       0.00      0.00      0.00         8
                            label_nose_too_high       0.00      0.00      0.00         8
                             label_nose_too_low       0.00      0.00      0.00         8
                         label_mouth_misaligned       0.26      0.88      0.40         8
                           label_mouth_too_high       0.00      0.00      0.00         8
                            label_mouth_too_low       0.00      0.00      0.00         8
label_nose_too_narrow_relative_to_inner_eye_gap       0.13      1.00      0.23         8
  label_nose_too_wide_relative_to_inner_eye_gap       0.00      0.00      0.00     